# Blindspot · Live Demo

End-to-end walkthrough of the **Blindspot** OpenEnv environment for unknown-unknowns discovery & contextual onboarding.

1. Boot the environment (in-process)
2. Reset for a real (held-out) user
3. Show the candidate pool
4. Run the **random**, **trending**, **dense-retrieval**, and **oracle** policies
5. Plot the reward gap (where RL has room to win)
6. Optional: drive the env with an LLM agent (`inference.py`)

In [ ]:
import sys, os, json
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve().parent))

from server.blindspot_environment import BlindspotEnvironment
from models import BlindspotAction

env = BlindspotEnvironment()
print(f'Loaded {len(env._data.user_ids)} users, {len(env._data.concept_catalog)} concepts.')

## 1. Reset for a real user

In [ ]:
obs = env.reset(seed=7, user_id=env._data.user_ids[0])
print('USER:', obs.user_id)
print('PROFILE:', obs.user_summary[:300], '...')
print(f'\n{len(obs.candidate_concepts)} candidates, inspect_budget={obs.inspect_budget_remaining}, surface_budget={obs.surface_budget_remaining}')
for c in obs.candidate_concepts[:5]:
    print(f'  id={c.concept_id}: {c.title}')

## 2. Inspect a concept (reveal the reading path)

In [ ]:
first_id = obs.candidate_concepts[0].concept_id
obs = env.step(BlindspotAction(type='inspect', concept_id=first_id))
detail = obs.inspected[str(first_id)]
print('Title:', detail.title)
print('Abstract:', detail.abstract_summary[:400])
print('\nReading path:')
for p in detail.top_papers:
    print(f'  - [{p["year"]}] {p["title"]}  (arxiv:{p["arxiv_id"]})')

## 3. Compare policies

In [ ]:
from baselines import random_baseline, trending_baseline, dense_retrieval_baseline
from scripts.make_plots import oracle_episode
import statistics

policies = {
    'random':           random_baseline.run_episode,
    'trending':         trending_baseline.run_episode,
    'dense_retrieval':  dense_retrieval_baseline.run_episode,
    'oracle (ceiling)': oracle_episode,
}
users = env._data.user_ids
for name, fn in policies.items():
    totals = [fn(env, uid, seed=s).get('total', 0.0) for uid in users for s in range(5)]
    print(f'{name:>20s}  mean={statistics.mean(totals):+.3f}  std={statistics.stdev(totals):.3f}')

## 4. False-positive calibration check

The FP penalty is calibrated so that random ≈ 0 (on the real precomputed dataset). Synthetic seed shows a small positive drift, but the gap to oracle (~+8.8) is what RL is trained to capture.

In [ ]:
from IPython.display import Image
Image('../plots/baseline_comparison.png')

## 5. Reward decomposition (per component)

In [ ]:
Image('../plots/reward_decomposition.png')

## 6. Optional: drive with an LLM agent

Uncomment to launch the LLM agent loop against a running env (start `uvicorn server.app:app --port 8000` in another shell first).

In [ ]:
# %env API_BASE_URL=https://api.openai.com/v1
# %env MODEL_NAME=gpt-4o-mini
# %env HF_TOKEN=...
# !python ../inference.py